In [ ]:
esm_model, alphabet = esm.pretrained.esm1b_t33_650M_UR50S()
batch_converter = alphabet.get_batch_converter()
esm_model.eval()

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm1b_t33_650M_UR50S.pt" to /root/.cache/torch/hub/checkpoints/esm1b_t33_650M_UR50S.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm1b_t33_650M_UR50S-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm1b_t33_650M_UR50S-contact-regression.pt


ProteinBertModel(
  (embed_tokens): Embedding(33, 1280, padding_idx=1)
  (layers): ModuleList(
    (0-32): 33 x TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
      )
      (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=1280, out_features=5120, bias=True)
      (fc2): Linear(in_features=5120, out_features=1280, bias=True)
      (final_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    )
  )
  (contact_head): ContactPredictionHead(
    (regression): Linear(in_features=660, out_features=1, bias=True)
    (activation): Sigmoid()
  )
  (embed_positions): LearnedPositionalEmbedding(1026, 1280, padding_idx=1)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import os
import glob
import torch
import pandas as pd
import numpy as np
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import jaccard_score
from tqdm.auto import tqdm
import esm

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# CONFIGURATION

DATA_BASE = '/content/drive/MyDrive/Data'
train_dir = f'{DATA_BASE}/train'
test_dir = f'{DATA_BASE}/test'
train_csv = f'{DATA_BASE}/train.csv'
emb_dir = f"{DATA_BASE}/esm_embeddings"


Using device: cuda


In [ ]:
# 2. EMBEDDING PRECOMPUTATION

def precompute_embeddings_efficiently():
    os.makedirs(emb_dir, exist_ok=True)
    model_name = "esm2_t6_8M_UR50D"
    esm_model, alphabet = esm.pretrained.__dict__[model_name]()
    batch_converter = alphabet.get_batch_converter()
    esm_model.eval()
    if torch.cuda.is_available():
        esm_model = esm_model.cuda()

    all_pdb_files = (
        glob.glob(os.path.join(train_dir, "*_protein.pdb")) +
        glob.glob(os.path.join(test_dir, "*_protein.pdb"))
    )
    print(f"Processing {len(all_pdb_files)} PDB files...")

    for pdb_path in tqdm(all_pdb_files):
        pdb_id = os.path.basename(pdb_path).replace("_protein.pdb", "")
        emb_path = os.path.join(emb_dir, f"{pdb_id}.npy")
        if os.path.exists(emb_path):
            continue

        try:
            sequence = extract_sequence_from_pdb(pdb_path)
            if not sequence or len(sequence) < 5:
                np.save(emb_path, np.zeros((0, 320)))
                continue

            if len(sequence) > 1000:
                sequence = sequence[:1000]

            data = [("protein", sequence)]
            _, _, tokens = batch_converter(data)
            if torch.cuda.is_available():
                tokens = tokens.cuda()

            with torch.no_grad():
                results = esm_model(tokens, repr_layers=[esm_model.num_layers])
                embedding = results["representations"][esm_model.num_layers][0, 1:-1]
                np.save(emb_path, embedding.cpu().numpy())

        except Exception as e:
            print(f"Error processing {pdb_id}: {e}")
            np.save(emb_path, np.zeros((0, 320)))

    del esm_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(" Embedding precomputation complete!")

def extract_sequence_from_pdb(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_path)
    sequences = {}
    for chain in structure.get_chains():
        seq = ""
        for residue in chain.get_residues():
            if residue.id[0] == ' ':
                try:
                    seq += seq1(residue.get_resname())
                except KeyError:
                    seq += "X"
        if seq:
            sequences[chain.id] = seq

    if not sequences:
        return None
    return max(sequences.values(), key=len)


In [ ]:
# 3. DATASET CLASS

class BindingPocketDataset(Dataset):
    def __init__(self, pdb_dir, emb_dir, labels_csv=None):
        self.pdb_paths = sorted(glob.glob(os.path.join(pdb_dir, "*_protein.pdb")))
        self.emb_dir = emb_dir
        self.is_train = labels_csv is not None
        if self.is_train:
            self.labels_df = pd.read_csv(labels_csv)
            self.labels_dict = {}
            for _, row in self.labels_df.iterrows():
                resids = str(row['resid']).split() if pd.notna(row['resid']) else []
                self.labels_dict[row['id']] = set(resids)

    def __len__(self):
        return len(self.pdb_paths)

    def __getitem__(self, idx):
        pdb_path = self.pdb_paths[idx]
        pdb_id = int(os.path.basename(pdb_path).replace("_protein.pdb", ""))
        emb_path = os.path.join(self.emb_dir, f"{pdb_id}.npy")

        if not os.path.exists(emb_path):
            seq_embedding = np.zeros((0, 320))
        else:
            seq_embedding = np.load(emb_path)

        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(str(pdb_id), pdb_path)
        coords = []
        residue_ids = []

        for residue in structure.get_residues():
            if residue.id[0] == ' ' and 'CA' in residue:
                coords.append(residue['CA'].get_coord())
                chain_id = residue.get_parent().id
                res_num = residue.id[1]
                residue_ids.append(f"{chain_id}_{res_num}")

        if len(coords) == 0:
            coords = np.zeros((0, 3))
            residue_ids = []
        else:
            coords = np.array(coords)

        min_len = min(len(coords), len(seq_embedding))
        if min_len == 0:
            features = np.zeros((1, 323))
            if self.is_train:
                labels = np.array([0], dtype=np.int64)
                return torch.tensor(features, dtype=torch.float32), torch.tensor(labels)
            else:
                return torch.tensor(features, dtype=torch.float32), pdb_id, ["A_1"]

        coords = coords[:min_len]
        seq_embedding = seq_embedding[:min_len]
        residue_ids = residue_ids[:min_len]
        features = np.concatenate([coords, seq_embedding], axis=1)

        if self.is_train:
            true_residues = self.labels_dict.get(pdb_id, set())
            labels = np.array([1 if res_id in true_residues else 0 for res_id in residue_ids], dtype=np.int64)
            return torch.tensor(features, dtype=torch.float32), torch.tensor(labels)
        else:
            return torch.tensor(features, dtype=torch.float32), pdb_id, residue_ids

In [ ]:
# 4. The MODEL

class ImprovedPocketModel(nn.Module):
    def __init__(self, in_dim=323):
        super().__init__()
        self.feature_net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.feature_net(x)

In [ ]:

# OPTIMIZATION

!mkdir -p /content/local_data
!cp -r "{train_dir}" /content/local_data/train
!cp -r "{test_dir}" /content/local_data/test
!cp -r "{emb_dir}" /content/local_data/esm_embeddings

train_dir = '/content/local_data/train'
test_dir = '/content/local_data/test'
emb_dir = '/content/local_data/esm_embeddings'


mydrive


In [ ]:
# 5. TRAINING FUNCTIONS

def collate_fn(batch):
    if len(batch[0]) == 2:  # Training
        features_list, labels_list = zip(*batch)
        return list(features_list), list(labels_list)
    else:  # Testing
        features_list, pdb_ids, residue_ids_list = zip(*batch)
        return list(features_list), list(pdb_ids), list(residue_ids_list)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for features_batch, labels_batch in tqdm(loader, desc="Training"):
        total_batch_loss = 0
        for features, labels in zip(features_batch, labels_batch):
            features = features.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_batch_loss += loss.item()
        total_loss += total_batch_loss / len(features_batch)
    return total_loss / len(loader)

def evaluate_model(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for features_batch, labels_batch in tqdm(loader, desc="Evaluating"):
            for features, labels in zip(features_batch, labels_batch):
                features = features.to(device)
                outputs = model(features)
                preds = torch.argmax(outputs, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())
    return jaccard_score(all_labels, all_preds, average='binary', zero_division=0)

In [ ]:
# 6. MAIN TRAINING

def main(resume_checkpoint=None):
    torch.manual_seed(42)
    np.random.seed(42)

    #  embeddings
    if not os.path.exists(emb_dir) or len(os.listdir(emb_dir)) < 100:
        print("Precomputing embeddings...")
        precompute_embeddings_efficiently()

    # Load data
    print("Loading datasets...")
    train_dataset = BindingPocketDataset(train_dir, emb_dir, train_csv)

    # Split
    train_size = int(0.8 * len(train_dataset))
    val_size = len(train_dataset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

    # loaders
    train_loader = DataLoader(train_subset, batch_size=4, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_subset, batch_size=4, shuffle=False, collate_fn=collate_fn)
    print(f"Training on {len(train_subset)} samples, validating on {len(val_subset)} samples")

    # Initialization
    model = ImprovedPocketModel().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

    # Resume
    start_epoch = 0
    best_iou = 0
    total_epochs = 8

    if resume_checkpoint and os.path.exists(resume_checkpoint):
        print(f" Resuming from checkpoint: {resume_checkpoint}")

        from numpy import _core
        torch.serialization.add_safe_globals([_core.multiarray.scalar])

        checkpoint = torch.load(resume_checkpoint, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch']
        best_iou = checkpoint['best_iou']
        print(f"↻ Resuming at epoch {start_epoch} with best IoU {best_iou:.4f}")
    else:
        print(" Starting training from scratch")

    # Training loop
    for epoch in range(start_epoch, total_epochs):
        print(f"\nEpoch {epoch+1}/{total_epochs}")
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_iou = evaluate_model(model, val_loader, device)
        print(f"Train Loss: {train_loss:.4f}, Val IoU: {val_iou:.4f}")

        if val_iou > best_iou:
            best_iou = val_iou
            torch.save(model.state_dict(), f'{DATA_BASE}/best_model.pth')
            print(f" New best IoU: {best_iou:.4f}")

        scheduler.step(train_loss)

        # Save checkpoint
        checkpoint_path = f"{DATA_BASE}/checkpoint_epoch_{epoch+1}.pth"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_iou': best_iou,
            'scheduler_state_dict': scheduler.state_dict(),
        }, checkpoint_path)
        print(f" Checkpoint saved at {checkpoint_path}")

    print(f" Training complete! Best IoU: {best_iou:.4f}")
    return model

In [ ]:
# 7. PREDICTION FUNCTION

def create_submission(model, test_dir, emb_dir, device):
    test_dataset = BindingPocketDataset(test_dir, emb_dir)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
    model.eval()
    submission_data = []

    with torch.no_grad():
        for features_batch, pdb_ids, residue_ids_batch in tqdm(test_loader, desc="Predicting"):
            for features, pdb_id, residue_ids in zip(features_batch, pdb_ids, residue_ids_batch):
                features = features.to(device)
                outputs = model(features)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                predicted_residues = [residue_ids[i] for i in range(len(preds)) if preds[i] == 1]
                submission_data.append({
                    'id': pdb_id,
                    'prediction': ' '.join(predicted_residues) if predicted_residues else ''
                })

    submission_df = pd.DataFrame(submission_data)
    submission_df.to_csv('submission.csv', index=False)
    print(" Submission file created: submission.csv")


In [ ]:
# 8. EXECUTION BLOCK
if __name__ == "__main__":
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    # Find latest checkpoint
    checkpoint_files = []
    if os.path.exists(DATA_BASE):
        checkpoint_files = [f for f in os.listdir(DATA_BASE)
                          if f.startswith('checkpoint_epoch') and f.endswith('.pth')]

    if checkpoint_files:
        checkpoint_files.sort(key=lambda x: int(x.split('_')[2].split('.')[0]))
        latest_checkpoint = os.path.join(DATA_BASE, checkpoint_files[-1])
        print(f" Found checkpoint: {latest_checkpoint}")
        trained_model = main(resume_checkpoint=latest_checkpoint)
    else:
        print(" No checkpoints found - starting fresh")
        trained_model = main()

    # Create submission
    create_submission(trained_model, test_dir, emb_dir, device)